<a href="https://colab.research.google.com/github/smsag99/Thesis/blob/main/codes/FeatureEng.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load Animal Data


Load the dataset from '/content/drive/MyDrive/Thesis_Data/Dati_ok.txt' into a pandas DataFrame.


In [1]:
datasetAnimalPath = '/content/drive/MyDrive/Thesis_Data/Dati_ok.txt'

In [50]:
import pandas as pd

# 1. Define your column names
col_names = [
    'Farm_Code', 'Animal_ID', 'dtb', 'dtc', 'dtt',
    'parity', 'milk_kg', 'fat_p', 'protein_p', 'cells', 'NM'
]

cleaned_rows = []

# 2. Read and clean the file line by line
with open(datasetAnimalPath, 'r') as file:
    # Skip the header line
    next(file)

    for line in file:
        # Remove whitespace/newlines from ends
        line = line.strip()

        # Skip empty lines if any exist
        if not line:
            continue

        # Split the line by comma
        parts = line.split(',')

        # LOGIC: Fix the rows based on length
        if len(parts) == 11:
            # This is a correct row, keep it as is
            cleaned_rows.append(parts)

        elif len(parts) == 12:
            # This is a broken row.
            # This leaves one empty slot for 'dtb' and shifts the rest back.
            parts.pop(2)
            cleaned_rows.append(parts)

# 3. Create the DataFrame
df_animal = pd.DataFrame(cleaned_rows, columns=col_names)

# 4. Convert columns to correct types (since they were read as strings)
numeric_cols = ['Farm_Code','parity', 'milk_kg', 'fat_p', 'protein_p', 'cells', 'NM']
for col in numeric_cols:
    df_animal[col] = pd.to_numeric(df_animal[col], errors='coerce')

# 5. Convert Date Columns
date_cols = ['dtb', 'dtc', 'dtt']
for col in date_cols:
    df_animal[col] = pd.to_datetime(df_animal[col], format='%Y%m%d', errors='coerce')

# Check the result
print(df_animal.head(5))
print(f"\nTotal rows loaded: {len(df_animal)}")

   Farm_Code       Animal_ID        dtb        dtc        dtt  parity  \
0     521513  IT003990094624 2013-10-29 2022-06-07 2022-06-20       6   
1     521513  IT003990094624 2013-10-29 2022-06-07 2022-07-19       6   
2     521513  IT003990094624 2013-10-29 2022-06-07 2022-09-12       6   
3     521513  IT003990094624 2013-10-29 2022-06-07 2022-10-13       6   
4     521513  IT003990094624 2013-10-29 2022-06-07 2022-11-15       6   

   milk_kg  fat_p  protein_p  cells  NM  
0      141    617        484    656   2  
1      168    682        417    827   2  
2      134    763        459   1144   2  
3      108    663        454   1403   2  
4       94    688        451   1805   2  

Total rows loaded: 2593267


### Scaling the Numeric Data

In [51]:
len(df_animal[df_animal['cells']==0])

118300

In [52]:
df_animal[df_animal['cells']==0] = 1

/tmp/ipykernel_3366/2044939430.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_animal[df_animal['cells']==0] = 1
/tmp/ipykernel_3366/2044939430.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_animal[df_animal['cells']==0] = 1
/tmp/ipykernel_3366/2044939430.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df_animal[df_animal['cells']==0] = 1


In [53]:
df_animal['milk_kg'] = df_animal['milk_kg'] /10
df_animal['fat_p'] = df_animal['fat_p'] / 100
df_animal['protein_p'] = df_animal['protein_p'] / 100
# df_animal['cells'] = df_animal['cells'] * 100000

### Adding ECM, AFC, SCS, DIM
#### Variable Calculations

**Energy-Corrected Milk (ECM)**
Buffalo-specific formula:
$$ECM = \left( \left[ \left( (Fat \times 10) - 40 \right) + \left( (Protein \times 10) - 31 \right) \right] \times 0.01155 + 1 \right) \times milk$$

**Somatic Cell Score (SCS)**
Conversion to linear score:
$$SCS = \log_{2}\left(\frac{Cells}{100}\right) + 3$$

**Age at First Calving (AFC)**
$$AFC = DTC - DTB$$

**Days in Milk (DIM)**
$$DIM = DTT - DTC$$

In [54]:
import numpy as np
df_animal['ECM'] = (((((df_animal['fat_p'] * 10) - 40) + ((df_animal['protein_p'] * 10) - 31)) * 0.01155) + 1) * df_animal['milk_kg']
df_animal['SCS'] = np.log2((df_animal['cells']) / 100) + 3
df_animal['AFC'] = df_animal['dtc'] - df_animal['dtb']
df_animal['DIM'] = df_animal['dtt'] - df_animal['dtc']

### Removing Outliers
####1. Component Thresholds
$$
\text{milk} =
\begin{cases}
\text{remove} & \text{if } \text{milk} \le 0.40 \text{ or } \text{milk} \ge 26.50 \\
\text{milk} & \text{otherwise}
\end{cases}
$$

$$
\text{fat} =
\begin{cases}
\text{remove} & \text{if } \text{fat} \le 1.60 \text{ or } \text{fat} \ge 15.05 \\
\text{fat} & \text{otherwise}
\end{cases}
$$

$$
\text{protein} =
\begin{cases}
\text{remove} & \text{if } \text{protein} \le 2.67 \text{ or } \text{protein} \ge 6.68 \\
\text{protein} & \text{otherwise}
\end{cases}
$$

$$
\text{SCS} =
\begin{cases}
\text{remove} & \text{if } \text{SCS} \le -2.06 \text{ or } \text{SCS} \ge 10.73 \\
\text{SCS} & \text{otherwise}
\end{cases}
$$

#### 2. ECM Logic
$$
\text{ECM} =
\begin{cases}
\text{remove} & \text{if } (\text{milk}, \text{fat}, \text{or protein}) = 0 \\
\text{remove} & \text{if } \text{ECM} \ge 30 \\
\text{ECM} & \text{otherwise}
\end{cases}
$$

#### 3. Inclusion Criteria (Days)
$$720 \le \text{AFC} \le 1440$$

In [66]:
df_animal_filtered = df_animal[(df_animal['milk_kg'] >= 0.4) & (df_animal['milk_kg'] <= 26.5)]
df_animal_filtered = df_animal_filtered[(df_animal_filtered['fat_p'] >= 1.6) & (df_animal_filtered['fat_p'] <= 15.05)]
df_animal_filtered = df_animal_filtered[(df_animal_filtered['protein_p'] >= 2.67) & (df_animal_filtered['protein_p'] <= 6.68)]
df_animal_filtered = df_animal_filtered[(df_animal_filtered['SCS'] >= -2.06) & (df_animal_filtered['SCS'] <= 10.73)]
df_animal_filtered = df_animal_filtered[df_animal_filtered['ECM'] <= 301]
df_animal_filtered = df_animal_filtered[(df_animal_filtered['AFC'] >= pd.Timedelta(days=720)) & (df_animal_filtered['AFC'] <= pd.Timedelta(days=1440))]

In [65]:
(len(df_animal)-len(df_animal_filtered))

1866833

## Loading Farm data
